In [0]:
from datetime import datetime, timezone
import hashlib

from pyspark.sql import functions as F

# Unity Catalog configuration.
CATALOG = "bucharest_mobility"
BRONZE = "bronze"

# Managed Volume containing the extracted GTFS Static source files.
SOURCE_PATH = f"/Volumes/{CATALOG}/{BRONZE}/raw_files"

# GTFS entities included in the static ingestion pipeline.
GTFS_FILES = [
    "agency",
    "stops",
    "routes",
    "calendar",
    "trips",
    "stop_times",
    "shapes",
]

In [0]:
def calculate_snapshot_id():


    """
    Generate a deterministic SHA-256 identifier for the complete
    GTFS Static snapshot.

    The hash includes both file names and file contents, so the same
    collection of GTFS files always produces the same snapshot ID.
    """

    hasher = hashlib.sha256()

    for file_name in sorted(GTFS_FILES):
        path = f"{SOURCE_PATH}/{file_name}.txt"

        file_bytes = (
            spark.read
            .format("binaryFile")
            .load(path)
            .select("content")
            .first()["content"]
        )

        hasher.update(file_name.encode())
        hasher.update(bytes(file_bytes))

    return hasher.hexdigest()
    

# One snapshot ID is shared by all GTFS entities from the same source state.

snapshot_id = calculate_snapshot_id()
ingested_at = datetime.now(timezone.utc)

print(f"Snapshot ID: {snapshot_id}")
print(f"Ingested at: {ingested_at}")

In [0]:
for table_name in GTFS_FILES:
    target_table = f"{CATALOG}.{BRONZE}.{table_name}"

    # Bronze keeps historical snapshots. If this exact source snapshot
    # already exists, skip it instead of creating duplicate history.
    if spark.catalog.tableExists(target_table):
        already_loaded = (
            spark.table(target_table)
            .filter(F.col("_snapshot_id") == snapshot_id)
            .limit(1)
            .count()
            > 0
        )
    else:
        already_loaded = False

    if already_loaded:
        print(f"SKIP {table_name}: snapshot already loaded")
        continue

    # Keep source columns as strings in Bronze to preserve the GTFS
    # representation. Typing and validation are handled in Silver.
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{SOURCE_PATH}/{table_name}.txt")
        .withColumn("_snapshot_id", F.lit(snapshot_id))
        .withColumn("_ingested_at", F.lit(ingested_at))
    )

    # Append-only Delta ingestion preserves historical GTFS snapshots.
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"LOADED {table_name}")

In [0]:
# Validate that each Bronze table is populated and report
# the number of historical snapshots currently retained.
for table_name in GTFS_FILES:
    df = spark.table(f"{CATALOG}.{BRONZE}.{table_name}")

    row_count = df.count()
    snapshot_count = (
        df.select("_snapshot_id")
        .distinct()
        .count()
    )

    print(
        f"{table_name}: "
        f"{row_count:,} rows | "
        f"{snapshot_count} snapshot(s)"
    )